# 11 · Earth Data Hub: terrain without a full download

**Objective:** Open Copernicus GLO-30 lazily, plot a small Finnish elevation subset and query a nearest grid value.

**Route:** Live remote Zarr / EDH API key · **Audience:** researcher / developer · **Duration:** 15–30 minutes after setup.

[Notebook index](README.md) · [Environment setup](../docs/setup/README.md) · [Accounts and API keys](../docs/setup/accounts.md) · [Data quality](../docs/reference/data-quality.md) · [Guide home](../README.md)

**Verification:** authored from the original workflow and official specifications; live access and your input files have **not been tested in this guide**. All distributed code cells have empty outputs. The release verification report records the separate offline checks. Run from a fresh kernel in numbered cell order.

**Expected result:** a small elevation map and a labelled nearest-grid query. Empty searches or missing permissions are possible; they are not evidence of a scientific result.

[Service overview and access route](../docs/services/earth-data-hub.md)


## 1. Load the local environment

Install the documented environment before opening this notebook. Paths in `.env` are relative to the repository root. Authentication is kept outside committed notebooks.


In [ ]:
from pathlib import Path
import sys
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dotenv import load_dotenv

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "examples" / "common.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Start Jupyter inside DestinE-partner-guide.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
load_dotenv(ROOT / ".env", override=False)
from examples.common import required_env, input_path, public_url, record_provenance


## 2. Select the current catalogue endpoint

Use the same private EDH netrc setup as notebook 09. Set `EDH_DEM_URL` to the current credential-free Copernicus GLO-30 catalogue access snippet. Metadata loading is small, but a country-scale 30 m `.load()` is expensive. This notebook computes only a roughly 1 km-scale AOI.


In [ ]:
# Read the request and account prerequisites before changing this switch.
RUN_LIVE = False
if not RUN_LIVE:
    raise RuntimeError("Live retrieval is disabled. Complete account setup, review quotas, then set RUN_LIVE=True.")


In [ ]:
import xarray as xr
from examples.climate import coordinate_slice
url = public_url(required_env("EDH_DEM_URL"))
ds = xr.open_dataset(url, engine="zarr", chunks={}, decode_coords="all", storage_options={"client_kwargs": {"trust_env": True}})
if "dsm" not in ds:
    raise ValueError("The catalogue schema changed; select its documented elevation variable.")
lat_name = "lat" if "lat" in ds.coords else "latitude"
lon_name = "lon" if "lon" in ds.coords else "longitude"
print({"metadata": ds.dsm.attrs, "sizes": dict(ds.sizes)})
# Inspect the Finland rectangle lazily; do not compute the whole country.
finland_metadata = ds.dsm.sel({lat_name: coordinate_slice(ds[lat_name], 59.5, 70.5), lon_name: coordinate_slice(ds[lon_name], 19, 32)})
print({"finland_subset_dimensions": dict(finland_metadata.sizes), "computed": False})
small = ds.dsm.sel({lat_name: coordinate_slice(ds[lat_name], 60.15, 60.16), lon_name: coordinate_slice(ds[lon_name], 24.90, 24.91)})
if small.size > 1_000_000:
    raise ValueError("Unexpectedly large subset; inspect coordinates.")
elevation = small.compute()
assert np.isfinite(elevation.values).any()
elevation.plot(figsize=(7, 4))
plt.title("Copernicus GLO-30 DSM — small Helsinki-area subset")
plt.show()


## 3. Query a nearby location and retain coordinates

Nearest-neighbour selection returns a grid cell, not a surveyed ground point. Record the actual grid coordinates and source vertical metadata.


In [ ]:
point = ds.dsm.sel({lat_name: 60.155, lon_name: 24.905}, method="nearest").compute()
print({"requested_lon_lat": [24.905, 60.155], "grid_lon": float(point[lon_name]), "grid_lat": float(point[lat_name]), "elevation": float(point.values), "units": ds.dsm.attrs.get("units")})
record_provenance("edh-dem.json", dataset_url=url, variable="dsm", variable_metadata=ds.dsm.attrs, subset_wgs84=[24.90, 60.15, 24.91, 60.16], actual_grid_point=[float(point[lon_name]), float(point[lat_name])])


## Interpret and verify

GLO-30 is a digital surface model: vegetation/buildings can affect the represented surface. Thirty-metre terrain does not resolve shallow peatland ditches or centimetre-scale water levels. Verify vertical datum and compare with finer local terrain for hydrology.

Check CRS, dates, masks, units, spatial extent, and valid sample count before comparing results. Keep product IDs and request parameters with the exported output. Do not treat synthetic/offline checks as verification of a remote service.


## Official references

- [Earth Data Hub setup and authentication](https://earthdatahub.destine.eu/getting-started)
- [Earth Data Hub catalogue](https://earthdatahub.destine.eu/)
- [Copernicus DEM product information](https://dataspace.copernicus.eu/explore-data/data-collections/copernicus-contributing-missions/collections-description/COP-DEM)

[Return to notebook index](README.md) · [Continue through Start here](../docs/start-here.md) · [Guide home](../README.md)
